# Bring Your Own Data

This notebook shows how to load **your actual data** into SEOCHO.

Three paths depending on your data format:
- **Path A**: Text files (.txt, .md)
- **Path B**: CSV / spreadsheet
- **Path C**: JSON / API responses

In [ ]:
from seocho import Seocho, Ontology, NodeDef, RelDef, P
from seocho.store import Neo4jGraphStore, OpenAIBackend
import json

# Adjust these to your environment
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"
DATABASE = "my_graph"
MODEL = "gpt-4o"

## Step 1: Define Your Ontology

**Change this to match your domain.** What entities exist? How do they connect?

In [ ]:
# EDIT THIS for your domain
ontology = Ontology(
    name="my_domain",
    nodes={
        # Add your entity types here
        "Person":  NodeDef(properties={"name": P(str, unique=True), "role": P(str)}),
        "Company": NodeDef(properties={"name": P(str, unique=True), "sector": P(str)}),
        # "Product": NodeDef(properties={"name": P(str, unique=True), "price": P(float)}),
    },
    relationships={
        # Add your relationships here
        "WORKS_AT": RelDef(source="Person", target="Company", cardinality="MANY_TO_ONE"),
        # "MAKES": RelDef(source="Company", target="Product"),
    },
)

print(f"Ontology: {ontology.name}")
print(f"  {len(ontology.nodes)} entity types: {', '.join(ontology.nodes.keys())}")
print(f"  {len(ontology.relationships)} relationships: {', '.join(ontology.relationships.keys())}")

In [ ]:
# Save your ontology (version control this file)
ontology.to_jsonld("my_schema.jsonld")
print("Saved to my_schema.jsonld")

In [ ]:
# Connect
s = Seocho(
    ontology=ontology,
    graph_store=Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD),
    llm=OpenAIBackend(model=MODEL),
)
s.ensure_constraints(database=DATABASE)
print("Connected and constraints applied")

---
## Path A: Text Files (.txt, .md)

Put your text files in a folder and index the whole directory.

In [ ]:
# Create sample data (replace with your actual data directory)
from pathlib import Path

data_dir = Path("./sample_data")
data_dir.mkdir(exist_ok=True)

(data_dir / "article1.txt").write_text("Tim Cook is the CEO of Apple. Apple is headquartered in Cupertino.")
(data_dir / "article2.txt").write_text("Jensen Huang founded NVIDIA in 1993. NVIDIA specializes in GPUs.")
(data_dir / "article3.txt").write_text("Satya Nadella leads Microsoft. Microsoft acquired GitHub in 2018.")

print(f"Created {len(list(data_dir.glob('*.txt')))} sample files")

In [ ]:
# Index the directory
result = s.index_directory(str(data_dir), database=DATABASE)

print(f"Files indexed: {result['files_indexed']}")
print(f"Files unchanged: {result['files_unchanged']}")
print(f"Files failed: {result['files_failed']}")

---
## Path B: CSV

Each row becomes a document. If your CSV has a `content` column, that's used as the text.

In [ ]:
# Create a sample CSV
import csv

csv_path = data_dir / "employees.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["content", "source"])
    writer.writerow(["Alice Chen is a Senior Engineer at Apple.", "hr_system"])
    writer.writerow(["Bob Kim manages the AI team at Samsung.", "hr_system"])
    writer.writerow(["Carol Lee is VP of Engineering at NVIDIA.", "hr_system"])

print(f"Created {csv_path}")

In [ ]:
# Index the CSV
result = s.index_file(str(csv_path), database=DATABASE)
print(f"Status: {result['status']}")
print(f"Records found: {result['records_found']}")

---
## Path C: JSON / API Responses

In [ ]:
# Create a sample JSON
json_path = data_dir / "api_data.json"
json_path.write_text(json.dumps([
    {"content": "Google announced Gemini 2.0 at I/O 2025.", "source": "news_api"},
    {"content": "Meta released Llama 4 with 1T parameters.", "source": "news_api"},
]))

result = s.index_file(str(json_path), database=DATABASE)
print(f"Status: {result['status']}, Records: {result['records_found']}")

---
## Query Your Data

In [ ]:
# Simple question
print(s.ask("Who works at Apple?", database=DATABASE))

In [ ]:
# With reasoning mode
print(s.ask(
    "What tech companies are mentioned and who leads them?",
    database=DATABASE,
    reasoning_mode=True,
    repair_budget=2,
))

In [ ]:
# Check graph status
records = s.query("MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count", database=DATABASE)
for r in records:
    print(f"  {r['type']}: {r['count']} nodes")

In [ ]:
s.close()
print("Done.")